# How to use the tiny-pointer GPU hash-join engine

A packaged, callable Python API (`hash_join.py` + `libhashjoin.so`) around the reliable
2-level tiny-pointer table — the same `bind(c)` + ctypes + CuPy bridge pattern
`../../MPDOK/mpdok_ops.py` already uses for its own solvers, not a new convention.

**Honest scope, up front**: this wins over a plain hash map only at HIGH load factor
(>=~95%) on a memory-tight, build-once/probe-many table (see `README.md` for the full
benchmark). Below that, a standard hash map is simpler and just as fast. Use this when you
specifically need a compact, high-load-factor, GPU-resident dictionary with a
guaranteed-zero build-failure rate — not as a general drop-in replacement for `dict`/
`cudf.merge`/`cuco::static_map`.

Three examples below, in increasing realism:
1. The minimal customers/orders example (correctness, by hand)
2. A larger random-key correctness + failure-detection check
3. A "what happens if you undersize the overflow" example — a real failure mode, shown
   honestly rather than only the happy path

In [1]:
import sys
sys.path.insert(0, ".")
import numpy as np
import cupy as cp
from hash_join import HashJoinTable

print("hash_join module loaded, libhashjoin.so linked OK")

hash_join module loaded, libhashjoin.so linked OK


## Example 1 — the minimal case: `customers JOIN orders`

Build a tiny table of 8 customers (customer_id -> loyalty tier via a row-id payload),
probe with 12 orders including 2 that reference customers who don't exist.

In [2]:
customer_id = cp.asarray([101, 102, 103, 104, 105, 106, 107, 108], dtype=cp.int64)
loyalty_tier = [3, 1, 2, 3, 1, 1, 2, 3]           # indexed by row id 0..7
rowid = cp.asarray(list(range(8)), dtype=cp.int32)

with HashJoinTable(n_keys=8, loadfactor=0.70, ovf_frac=0.25, B=4) as table:
    table.build(customer_id, rowid)

    orders = cp.asarray([101, 105, 977, 103, 108, 101, 988, 106, 102, 104, 107, 108], dtype=cp.int64)
    matched_rowid = cp.asnumpy(table.probe(orders))

    for cust, r in zip(cp.asnumpy(orders), matched_rowid):
        tier = loyalty_tier[r] if r >= 0 else "NO MATCH"
        print(f"  order for customer {cust:>4} -> tier {tier}")

  order for customer  101 -> tier 3
  order for customer  105 -> tier 1
  order for customer  977 -> tier NO MATCH
  order for customer  103 -> tier 2
  order for customer  108 -> tier 3
  order for customer  101 -> tier 3
  order for customer  988 -> tier NO MATCH
  order for customer  106 -> tier 1
  order for customer  102 -> tier 1
  order for customer  104 -> tier 3
  order for customer  107 -> tier 2
  order for customer  108 -> tier 3


## Example 2 — a realistic scale: 200,000 random keys

The shape this engine is actually built for: a large, build-once/probe-many dictionary of
distinct keys (e.g. a feature-store ID lookup, a deduplication set, a dimension-table join
key), at a genuinely high load factor.

In [3]:
rng = np.random.default_rng(0)
n = 200_000
keys_np = rng.choice(np.arange(1, 10_000_000, dtype=np.int64), size=n, replace=False)
keys = cp.asarray(keys_np)
values = cp.asarray(np.arange(n, dtype=np.int32))

with HashJoinTable(n_keys=n, loadfactor=0.90, ovf_frac=0.15) as table:
    table.build(keys, values)   # raises RuntimeError if any key failed -- it won't here

    # probe with a mix of real keys (should match) and definitely-absent keys (should not)
    real_subset = keys[:1000]
    fake_keys = cp.asarray(rng.choice(np.arange(20_000_000, 30_000_000, dtype=np.int64),
                                      size=1000, replace=False))
    probe_keys = cp.concatenate([real_subset, fake_keys])
    result = cp.asnumpy(table.probe(probe_keys))

    real_correct = (result[:1000] == cp.asnumpy(values[:1000])).all()
    fake_all_nomatch = (result[1000:] == -1).all()
    print(f"{n:,} keys built with 0 failures.")
    print(f"1,000 real-key probes all correct: {bool(real_correct)}")
    print(f"1,000 fake-key probes all correctly NO MATCH: {bool(fake_all_nomatch)}")

200,000 keys built with 0 failures.
1,000 real-key probes all correct: True
1,000 fake-key probes all correctly NO MATCH: True


## Example 3 — an honest failure mode: undersizing the overflow region

`build()` raises, rather than silently dropping keys, if the overflow region is too small
for the actual key distribution. This is deliberate: a caller should never have to guess
whether their table is missing data.

In [4]:
# A deliberately-undersized table: ovf_frac=0.0 means no overflow backup at all, so
# ANY bucket collision beyond capacity B is a hard failure -- a real, honest demonstration
# of the failure mode README.md's "6% worst-case random-key spill rate" note warns about.
n_small = 5_000
keys_small = cp.asarray(rng.choice(np.arange(1, 1_000_000, dtype=np.int64), size=n_small, replace=False))
values_small = cp.asarray(np.arange(n_small, dtype=np.int32))

try:
    with HashJoinTable(n_keys=n_small, loadfactor=0.90, ovf_frac=0.0, B=16) as table:
        table.build(keys_small, values_small)
        print("build succeeded (no collisions this run -- possible but not guaranteed at ovf_frac=0)")
except RuntimeError as e:
    print(f"build() correctly raised instead of silently dropping keys:\n  {e}")

build() correctly raised instead of silently dropping keys:
  HashJoinTable.build: 245/5000 keys failed to insert (table sized for n_keys=5000; either this build has more keys than that, or ovf_frac is too small for this key distribution — see README.md's honest note on random vs. sequential key spill rates)


## Summary

- **The packaged API works exactly like the raw CUDA Fortran demo** (`join_example_small.cuf`)
  — same correctness on the same toy data, now callable from ordinary Python/CuPy code
  instead of a compiled CLI benchmark binary.
- **It scales to a realistic size** (200,000 random keys) with zero build failures at 90%
  load, and correctly distinguishes real matches from genuinely absent keys.
- **Failure is loud, not silent** — an undersized overflow region raises `RuntimeError`
  from `build()` rather than quietly dropping keys, so a caller always knows whether their
  table is trustworthy.
- **Remember the honest scope**: this is a niche tool for high-load-factor, memory-tight,
  build-once/probe-many dictionaries — not a general hash-map replacement. See `README.md`
  for the full benchmark showing exactly where the crossover against plain linear probing
  sits (~95% load factor).